In [ ]:
from zipfile import ZipFile
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler

In [ ]:
path = "/content/house-prices-advanced-regression-techniques.zip"
zip_obj = ZipFile(path, "r")
zip_obj.extractall("./")
zip_obj.close()

In [ ]:
train_data = pd.read_csv("/content/train.csv")
test_data = pd.read_csv("/content/test.csv")

In [ ]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [ ]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1459 non-null   int64  
 1   MSSubClass     1459 non-null   int64  
 2   MSZoning       1455 non-null   object 
 3   LotFrontage    1232 non-null   float64
 4   LotArea        1459 non-null   int64  
 5   Street         1459 non-null   object 
 6   Alley          107 non-null    object 
 7   LotShape       1459 non-null   object 
 8   LandContour    1459 non-null   object 
 9   Utilities      1457 non-null   object 
 10  LotConfig      1459 non-null   object 
 11  LandSlope      1459 non-null   object 
 12  Neighborhood   1459 non-null   object 
 13  Condition1     1459 non-null   object 
 14  Condition2     1459 non-null   object 
 15  BldgType       1459 non-null   object 
 16  HouseStyle     1459 non-null   object 
 17  OverallQual    1459 non-null   int64  
 18  OverallC

In [ ]:
drop = ["Alley", "MasVnrType", "FireplaceQu", "PoolQC", "Fence", "MiscFeature"]
train_data.drop(drop, axis = 1, inplace=True)
test_data.drop(drop, axis = 1, inplace=True)

In [ ]:
train_data.drop("Id", axis=1, inplace=True)

In [ ]:
def fill_missing_values(data):
  for column in data.columns:
      if data[column].dtype == "object":
          mode_value = data[column].mode()[0]
          data[column].fillna(mode_value, inplace=True)
      elif data[column].dtype == "float":
          mean_value = data[column].mean()
          data[column].fillna(mean_value, inplace=True)
  return data

In [ ]:
train_data = fill_missing_values(train_data)
test_data = fill_missing_values(test_data)

In [ ]:
size =  len(test_data)
train_data = train_data[:size]

In [ ]:
def encode_categorical_columns(df):
    df_copy = df.copy()

    label_encoder = LabelEncoder()

    for column in df_copy.columns:
        if df_copy[column].dtype == "object":
            df_copy[column] = label_encoder.fit_transform(df_copy[column])

    return df_copy

In [ ]:
train_data = encode_categorical_columns(train_data)
test_data = encode_categorical_columns(test_data)

In [ ]:
index_column = train_data.columns.get_loc("SalePrice")
X = train_data.iloc[:, :index_column].values
y = train_data.iloc[:, index_column].values

In [ ]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
X_train.shape, X_test.shape # -> ((1021, 73), (438, 73))

((1021, 73), (438, 73))

In [ ]:
y_train.shape, y_test.shape # -> ((1021, 1), (438, 1))

((1021,), (438,))

In [ ]:
def house_model(units):
    model = tf.keras.Sequential([
        layers.Dense(units, activation="relu", input_shape=(73,)),
        layers.Dense(units, activation="relu"),
        layers.Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")

    return model

In [ ]:
EPOCHS = 100
UNITS = 32
model = house_model(UNITS)

In [ ]:
model.fit(X_train, y_train,
          epochs=EPOCHS,
          validation_data=(X_test, y_test),
          verbose=1)

Epoch 1/100
32/32 [==============================] - 3s 10ms/step - loss: 39251406848.0000 - val_loss: 38582554624.0000
Epoch 2/100
32/32 [==============================] - 0s 4ms/step - loss: 39249666048.0000 - val_loss: 38579740672.0000
Epoch 3/100
32/32 [==============================] - 0s 4ms/step - loss: 39244754944.0000 - val_loss: 38572130304.0000
Epoch 4/100
32/32 [==============================] - 0s 4ms/step - loss: 39233536000.0000 - val_loss: 38556786688.0000
Epoch 5/100
32/32 [==============================] - 0s 4ms/step - loss: 39212572672.0000 - val_loss: 38529073152.0000
Epoch 6/100
32/32 [==============================] - 0s 4ms/step - loss: 39176589312.0000 - val_loss: 38483734528.0000
Epoch 7/100
32/32 [==============================] - 0s 5ms/step - loss: 39120224256.0000 - val_loss: 38416572416.0000
Epoch 8/100
32/32 [==============================] - 0s 4ms/step - loss: 39039938560.0000 - val_loss: 38323195904.0000
Epoch 9/100
32/32 [============================

In [ ]:
ids = test_data["Id"]
ids

0       1461
1       1462
2       1463
3       1464
4       1465
        ... 
1454    2915
1455    2916
1456    2917
1457    2918
1458    2919
Name: Id, Length: 1459, dtype: int64

In [ ]:
test_data.drop("Id", axis=1, inplace=True)

In [ ]:
features = np.array(test_data)
features = scaler.fit_transform(features)
features

array([[0.        , 0.5       , 0.32960894, ..., 1.        , 1.        ,
        0.8       ],
       [0.        , 0.75      , 0.33519553, ..., 1.        , 1.        ,
        0.8       ],
       [0.23529412, 0.75      , 0.29608939, ..., 1.        , 1.        ,
        0.8       ],
       ...,
       [0.        , 0.75      , 0.77653631, ..., 0.        , 1.        ,
        0.        ],
       [0.38235294, 0.75      , 0.22905028, ..., 0.        , 1.        ,
        0.8       ],
       [0.23529412, 0.75      , 0.29608939, ..., 0.        , 1.        ,
        0.8       ]])

In [ ]:
predicitions = model.predict(features)
predicitions

46/46 [==============================] - 0s 2ms/step


array([[175525.86],
       [178041.14],
       [189244.16],
       ...,
       [186873.9 ],
       [174746.84],
       [193732.88]], dtype=float32)

In [ ]:
output = pd.DataFrame({'Id': ids,
                       'SalePrice': predicitions.squeeze()})
output.head()

,Id,SalePrice
0,1461,175525.859375
1,1462,178041.140625
2,1463,189244.156250
3,1464,192566.578125
4,1465,175408.718750


In [ ]:
sample = pd.read_csv("/content/sample_submission.csv")
sample.head()

,Id,SalePrice
0,1461,169277.052498
1,1462,187758.393989
2,1463,183583.683570
3,1464,179317.477511
4,1465,150730.079977


In [ ]:
abs((sample["SalePrice"] - output["SalePrice"]).mean())

512.3439837523431

In [ ]:
output.to_csv('House_Price_Forecast.csv', index=False)